# 03 — GPU KV Cache Allocation
**CUDA kernel-level block allocation from real GPU memory**

---

```
GPU memory budget (T4 ~15 GB)
│
├── Model weights          (fp16)   ~220 MB
├── Activations + overhead          ~500 MB  (estimated)
│
└── REMAINING  ──────────────────────────────────► KV cache pool
                                                    carved into blocks
                                                    of BLOCK_SIZE tokens
```

Instead of pre-sizing the pool with a formula, we:
1. Load the model onto GPU
2. Query `torch.cuda.mem_get_info()` for free memory
3. Reserve a safety margin (for activations during forward pass)
4. Allocate **one contiguous GPU tensor** for the entire KV pool
5. Write CUDA kernels (`.cu` compiled via `nvcc` + `ctypes`) to:
   - `kv_init_blocks` — zero-fill newly allocated blocks
   - `kv_write`       — scatter K/V into paged slots
   - `kv_read`        — gather K/V from paged slots
   - `kv_clear_blocks`— zero-fill freed blocks (reset slots)

### Memory layout in the contiguous pool
```
kv_pool_k : (N_BLOCKS, BLOCK_SIZE, N_KV_HEADS, HEAD_DIM)  fp16  [contiguous]
kv_pool_v : (N_BLOCKS, BLOCK_SIZE, N_KV_HEADS, HEAD_DIM)  fp16  [contiguous]

To write token at (block_id=7, slot=4, head=2, dim=*):
  kv_pool_k[7, 4, 2, :] = k_vector

CUDA kernel does this in parallel for all tokens in a batch.
```

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## 1. Install & GPU Check

In [2]:
!pip install -q sentencepiece transformers

import torch
import torch.nn as nn
import torch.nn.functional as F
import math, time, os, gc
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

assert torch.cuda.is_available(), 'Need GPU — set runtime to T4 in Colab'
device = 'cuda'

props = torch.cuda.get_device_properties(0)
TOTAL_VRAM = props.total_memory
print(f'GPU          : {props.name}')
print(f'Total VRAM   : {TOTAL_VRAM / 1e9:.2f} GB')
torch.manual_seed(42)
torch.backends.cuda.matmul.allow_tf32 = True

GPU          : Tesla T4
Total VRAM   : 15.64 GB


---
## 2. Load Model — Measure Baseline Memory

We load the LLaMA model from the checkpoint and record exactly how much GPU memory it occupies. Everything else is available for KV cache.

In [3]:
# ── Paste LLaMA backbone (same as notebooks 01/02) ───────────────────────────
from transformers import AutoTokenizer

@dataclass
class LLaMAConfig:
    vocab_size      : int   = 32000
    dim             : int   = 768
    n_layers        : int   = 12
    n_heads         : int   = 12
    n_kv_heads      : int   = 6
    max_seq_len     : int   = 512
    ffn_dim_mult    : float = 2.6875
    norm_eps        : float = 1e-5
    rope_theta      : float = 10000.0
    dropout         : float = 0.0
    grad_checkpoint : bool  = False
    # training fields for checkpoint compat
    batch_size      : int   = 16
    grad_accum_steps: int   = 4
    max_iters       : int   = 10000
    eval_interval   : int   = 500
    eval_iters      : int   = 50
    log_interval    : int   = 100
    learning_rate   : float = 3e-4
    weight_decay    : float = 0.1
    beta1           : float = 0.9
    beta2           : float = 0.95
    grad_clip       : float = 1.0
    warmup_iters    : int   = 200
    min_lr          : float = 3e-5

    @property
    def head_dim(self):
      return self.dim // self.n_heads
    @property
    def ffn_hidden_dim(self):
        return ((int(self.ffn_dim_mult * self.dim) + 255) // 256) * 256

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return self.gamma * (x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x)

def precompute_rope_freqs(head_dim, max_seq_len, theta=10000.0, device='cpu'):
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32, device=device) / head_dim))
    t = torch.arange(max_seq_len, dtype=torch.float32, device=device)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs).repeat_interleave(2, dim=-1), torch.sin(freqs).repeat_interleave(2, dim=-1)

def rotate_half(x):
    return torch.stack([-x[..., 1::2], x[..., 0::2]], dim=-1).flatten(-2)

def apply_rope(q, k, cos, sin):
    c, s = cos.unsqueeze(0).unsqueeze(0), sin.unsqueeze(0).unsqueeze(0)
    return (q*c + rotate_half(q)*s).type_as(q), (k*c + rotate_half(k)*s).type_as(k)

def repeat_kv(x, n_rep):
    if n_rep == 1: return x
    B, nkv, T, hd = x.shape
    return x[:,:,None,:,:].expand(B,nkv,n_rep,T,hd).reshape(B,nkv*n_rep,T,hd)

class SwiGLUFFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        h = cfg.ffn_hidden_dim
        self.w1 = nn.Linear(cfg.dim, h, bias=False)
        self.w2 = nn.Linear(h, cfg.dim, bias=False)
        self.w3 = nn.Linear(cfg.dim, h, bias=False)
    def forward(self, x): return self.w2(F.silu(self.w3(x)) * self.w1(x))

class LLaMAAttentionSimple(nn.Module):
    """Minimal attention — just weights, no cache. Used only for memory measurement."""
    def __init__(self, cfg, layer_idx):
        super().__init__()
        self.n_heads = cfg.n_heads; self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.head_dim; self.layer_idx = layer_idx
        self.wq = nn.Linear(cfg.dim, cfg.n_heads * cfg.head_dim, bias=False)
        self.wk = nn.Linear(cfg.dim, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.wv = nn.Linear(cfg.dim, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.wo = nn.Linear(cfg.n_heads * cfg.head_dim, cfg.dim, bias=False)

class LLaMABlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        self.attention_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.attention = LLaMAAttentionSimple(cfg, layer_idx)
        self.ffn_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.ffn = SwiGLUFFN(cfg)

class LLaMAModelWeightsOnly(nn.Module):
    """Loads only the weight tensors — no forward pass needed for memory measurement."""
    def __init__(self, cfg):
        super().__init__()
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.layers = nn.ModuleList([LLaMABlock(cfg, i) for i in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)
        self.token_emb.weight = self.lm_head.weight
    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())

print('Backbone classes defined')

Backbone classes defined


In [4]:
# ── Load checkpoint ──────────────────────────────────────────────────────────
CKPT = '/content/drive/MyDrive/llama_ckpt.pt'

torch.cuda.empty_cache()
gc.collect()

# Measure free memory BEFORE loading model
free_before, total = torch.cuda.mem_get_info()
print(f'Before model load:')
print(f'  Free  : {free_before/1e9:.3f} GB')
print(f'  Total : {total/1e9:.3f} GB')

# Load
ckpt = torch.load(CKPT, map_location='cpu', weights_only=True)
valid = {f for f in LLaMAConfig.__dataclass_fields__}
cfg   = LLaMAConfig(**{k:v for k,v in ckpt['config'].items() if k in valid})

cos_table, sin_table = precompute_rope_freqs(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta, device)

model = LLaMAModelWeightsOnly(cfg)
model.load_state_dict(ckpt['model_state'], strict=False)
model = model.half().to(device)   # fp16
model.eval()

torch.cuda.synchronize()

# Measure free memory AFTER loading model
free_after, _ = torch.cuda.mem_get_info()
model_vram = free_before - free_after

print(f'\nAfter model load:')
print(f'  Free      : {free_after/1e9:.3f} GB')
print(f'  Model uses: {model_vram/1e9:.3f} GB  ({model_vram/1e6:.1f} MB)')
print(f'  Params    : {model.num_parameters()/1e6:.1f}M  '
      f'(expect ~{model.num_parameters()*2/1e6:.0f} MB in fp16)')

Before model load:
  Free  : 15.527 GB
  Total : 15.637 GB

After model load:
  Free      : 15.273 GB
  Model uses: 0.254 GB  (253.8 MB)
  Params    : 109.5M  (expect ~219 MB in fp16)


---
## 3. Compute Available KV Memory

```
Free GPU memory after model load
  − activation_reserve   (for forward pass peaks)
  − safety_margin        (fragmentation, CUDA overhead)
  ─────────────────────────────────────────────────
  = kv_budget             → divide into blocks
```

### One physical block stores:
shape: (16, n_kv_heads, head_dim) = (16, 6, 64)  fp16

       └── 16 token slots
               └── 6 heads each
                       └── 64 dims each

This is EITHER K or V, for ONE layer only.

### So for one logical block (16 tokens), what do you actually need?
For each of 12 layers:
    K block → (16, 6, 64)   ← pool_k[phys_id_layer_0]
    V block → (16, 6, 64)   ← pool_v[phys_id_layer_0]

    K block → (16, 6, 64)   ← pool_k[phys_id_layer_1]
    V block → (16, 6, 64)   ← pool_v[phys_id_layer_1]
    ...
    K block → (16, 6, 64)   ← pool_k[phys_id_layer_11]
    V block → (16, 6, 64)   ← pool_v[phys_id_layer_11]

Total: 12 × 2 = 24 physical blocks

In this notebook pool_k and pool_v are separate tensors
So one physical block id indexes into BOTH pools:
    pool_k : (N_PHYSICAL_BLOCKS, 16, 6, 64)   ← all K blocks
    pool_v : (N_PHYSICAL_BLOCKS, 16, 6, 64)   ← all V blocks

    pool_k[7]  →  K for layer 3, logical block 0  of seq A
    pool_v[7]  →  V for layer 3, logical block 0  of seq A

    Token at logical pos 20 in seq A
            │
            ▼
    For each of 12 transformer layers:
        wk(x) → k vector  shape (6, 64)
        wv(x) → v vector  shape (6, 64)
            │
            ▼
        translate(layer, pos=20)
        → logical_block = 20//16 = 1
        → slot          = 20% 16 = 4
        → phys_id       = block_ids[layer][1]   ← different int per layer
            │
            ▼
        pool_k[phys_id, 4, :, :]  ← write k vector here   (6, 64)
        pool_v[phys_id, 4, :, :]  ← write v vector here   (6, 64)


In [5]:
# ── Memory budget ────────────────────────────────────────────────────────────

# Run a dummy forward pass to prime CUDA and measure peak activation memory
# This gives us a realistic activation footprint
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

# Safety margins
ACTIVATION_RESERVE_GB = 1.0    # headroom for forward pass activations
SAFETY_MARGIN_GB      = 0.5    # CUDA fragmentation + misc overhead

free_now, _ = torch.cuda.mem_get_info()

kv_budget_bytes = int(
    free_now
    - ACTIVATION_RESERVE_GB * 1e9
    - SAFETY_MARGIN_GB      * 1e9
)
kv_budget_bytes = max(kv_budget_bytes, 0)

print('=== GPU Memory Budget ===')
print(f'  Total VRAM              : {total/1e9:.3f} GB')
print(f'  Model weights (actual)  : {model_vram/1e9:.3f} GB')
print(f'  Free after model        : {free_now/1e9:.3f} GB')
print(f'  Activation reserve      : {ACTIVATION_RESERVE_GB:.1f} GB')
print(f'  Safety margin           : {SAFETY_MARGIN_GB:.1f} GB')
print(f'  ──────────────────────────────────')
print(f'  KV cache budget         : {kv_budget_bytes/1e9:.3f} GB  ({kv_budget_bytes/1e6:.0f} MB)')

# ── How many blocks fit? ─────────────────────────────────────────────────────
BLOCK_SIZE   = 16
DTYPE_BYTES  = 2      # fp16
KV           = 2      # K and V

# Bytes per logical block (one block_id = one layer's worth) stores K or V in cuda memory
bytes_per_physical_block = (
    BLOCK_SIZE * cfg.n_kv_heads * cfg.head_dim * DTYPE_BYTES
)
# pool_k needs: N_PHYSICAL_BLOCKS physical blocks
# pool_v needs: N_PHYSICAL_BLOCKS physical blocks
# so total memory = 2 × N_PHYSICAL_BLOCKS × bytes_per_physical_block

# We want: 2 × N_PHYSICAL_BLOCKS × bytes_per_physical_block <= budget
# → N_PHYSICAL_BLOCKS <= budget / (2 × bytes_per_physical_block)
# → N_LOGICAL_BLOCKS  <= budget / (2 × n_layers × bytes_per_physical_block)
# For ALL layers (K+V), one logical block costs:
bytes_per_logical_block = KV * cfg.n_layers * bytes_per_physical_block

# Number of logical blocks that fit in the budget
N_LOGICAL_BLOCKS = int(kv_budget_bytes // bytes_per_logical_block)

# Allocator tracks per-layer physical blocks
N_PHYSICAL_BLOCKS = N_LOGICAL_BLOCKS * cfg.n_layers  # total allocator slots

# Actual bytes we will allocate (round down to exact block boundary)
KV_POOL_BYTES = N_LOGICAL_BLOCKS * bytes_per_logical_block

print(f'\n=== KV Block Sizing ===')
print(f'  BLOCK_SIZE              : {BLOCK_SIZE} tokens')
print(f'  bytes per physical block: {bytes_per_physical_block} B')
print(f'  bytes per logical block : {bytes_per_logical_block/1e3:.2f} KB  '
      f'({cfg.n_layers} layers × K+V)')
print(f'  N_LOGICAL_BLOCKS        : {N_LOGICAL_BLOCKS}')
print(f'  N_PHYSICAL_BLOCKS       : {N_PHYSICAL_BLOCKS}  (allocator slots)')
print(f'  KV pool allocated       : {KV_POOL_BYTES/1e6:.1f} MB')
print(f'  Max tokens cached       : {N_LOGICAL_BLOCKS * BLOCK_SIZE:,}')

=== GPU Memory Budget ===
  Total VRAM              : 15.637 GB
  Model weights (actual)  : 0.254 GB
  Free after model        : 15.273 GB
  Activation reserve      : 1.0 GB
  Safety margin           : 0.5 GB
  ──────────────────────────────────
  KV cache budget         : 13.773 GB  (13773 MB)

=== KV Block Sizing ===
  BLOCK_SIZE              : 16 tokens
  bytes per physical block: 12288 B
  bytes per logical block : 294.91 KB  (12 layers × K+V)
  N_LOGICAL_BLOCKS        : 46703
  N_PHYSICAL_BLOCKS       : 560436  (allocator slots)
  KV pool allocated       : 13773.3 MB
  Max tokens cached       : 747,248


---
## 4. Allocate Contiguous KV Pool

Instead of a Python dict of tensors (notebook 02), we allocate
**one large contiguous GPU tensor** per K and V:

```
pool_k : (N_PHYSICAL_BLOCKS, BLOCK_SIZE, N_KV_HEADS, HEAD_DIM)  fp16
pool_v : same

Advantages:
  • Single cudaMalloc call — no fragmentation
  • Cache-friendly access patterns
  • CUDA kernels can index directly with integer arithmetic
  • Entire pool reserved upfront — no OOM mid-generation
```

In [6]:
torch.cuda.synchronize()
free_before_pool, _ = torch.cuda.mem_get_info()

# ── Allocate the two contiguous KV tensors ───────────────────────────────────
# Shape: (N_PHYSICAL_BLOCKS, BLOCK_SIZE, N_KV_HEADS, HEAD_DIM)
# Each physical block belongs to exactly one layer (LayeredBlockTable design)
POOL_SHAPE = (N_PHYSICAL_BLOCKS, BLOCK_SIZE, cfg.n_kv_heads, cfg.head_dim)

pool_k = torch.zeros(POOL_SHAPE, dtype=torch.float16, device=device)
pool_v = torch.zeros(POOL_SHAPE, dtype=torch.float16, device=device)

torch.cuda.synchronize()
free_after_pool, _ = torch.cuda.mem_get_info()
pool_actual_bytes = free_before_pool - free_after_pool

print('=== Contiguous KV Pool Allocated ===')
print(f'  pool_k shape  : {list(pool_k.shape)}')
print(f'  pool_v shape  : {list(pool_v.shape)}')
print(f'  pool_k stride : {list(pool_k.stride())}  (row-major, cache-friendly for block access)')
print(f'  Requested     : {KV_POOL_BYTES/1e6:.1f} MB')
print(f'  Actual used   : {pool_actual_bytes/1e6:.1f} MB')
print(f'  pool_k ptr    : 0x{pool_k.data_ptr():016x}')
print(f'  pool_v ptr    : 0x{pool_v.data_ptr():016x}')
ptr_gap = abs(pool_v.data_ptr() - pool_k.data_ptr())
print(f'  ptr gap K→V   : {ptr_gap/1e6:.1f} MB  (= pool_k size)')

# Verify memory is contiguous
assert pool_k.is_contiguous(), 'pool_k must be contiguous'
assert pool_v.is_contiguous(), 'pool_v must be contiguous'
print('\nBoth pools are contiguous in GPU memory')

# Final memory summary
free_final, _ = torch.cuda.mem_get_info()
print(f'\n=== Final GPU Memory State ===')
print(f'  Total               : {total/1e9:.3f} GB')
print(f'  Model weights       : {model_vram/1e6:.0f} MB')
print(f'  KV pool (K+V)       : {pool_actual_bytes/1e6:.0f} MB')
print(f'  Remaining free      : {free_final/1e6:.0f} MB  (activation buffer)')

=== Contiguous KV Pool Allocated ===
  pool_k shape  : [560436, 16, 6, 64]
  pool_v shape  : [560436, 16, 6, 64]
  pool_k stride : [6144, 384, 64, 1]  (row-major, cache-friendly for block access)
  Requested     : 13773.3 MB
  Actual used   : 13774.1 MB
  pool_k ptr    : 0x000078f262000000
  pool_v ptr    : 0x000078f0c6000000
  ptr gap K→V   : 6912.2 MB  (= pool_k size)

Both pools are contiguous in GPU memory

=== Final GPU Memory State ===
  Total               : 15.637 GB
  Model weights       : 254 MB
  KV pool (K+V)       : 13774 MB
  Remaining free      : 1499 MB  (activation buffer)


---
## 5. CUDA Kernels (pure C++/CUDA — nvcc + ctypes)

```
kv_kernels.cu
      │
      │  nvcc -O2 --use_fast_math -shared -Xcompiler -fPIC
      ▼
kv_ops.so   ← ctypes.CDLL(RTLD_GLOBAL)
      │
      ▼
Python wrappers
  tensor.data_ptr()  →  ctypes.c_void_p  →  void* in C
```

| Kernel | Grid | Block | Job |
|---|---|---|---|
| `kv_init_blocks` | `(N_BLOCKS, BLOCK_SIZE, N_KV_HEADS)` | `(HEAD_DIM,)` | Zero-fill blocks |
| `kv_write` | `(T, N_KV_HEADS)` | `(HEAD_DIM,)` | Scatter K/V into pool |
| `kv_read` | `(T, N_KV_HEADS)` | `(HEAD_DIM,)` | Gather K/V from pool |
| `kv_clear_blocks` | same as init | same | Zero freed blocks |

### Memory addressing (same in all kernels)
```c
// pool shape: (N_PHYS_BLOCKS, BLOCK_SIZE, N_KV_HEADS, HEAD_DIM)
int offset = phys_id * BLOCK_SIZE * N_KV_HEADS * HEAD_DIM
           + slot    * N_KV_HEADS * HEAD_DIM
           + head    * HEAD_DIM
           + dim;    // threadIdx.x
```

In [7]:
import subprocess, ctypes, os

CUDA_SRC = r"""
#include <cuda_runtime.h>
#include <cuda_fp16.h>
#include <stdint.h>          // ← needed for int64_t

__global__ void kv_init_blocks(
    __half* __restrict__ pool_k,
    __half* __restrict__ pool_v,
    const int* __restrict__ block_ids,
    int block_size, int n_kv_heads, int head_dim
) {
    int blk_idx = blockIdx.x;
    int slot    = blockIdx.y;
    int head    = blockIdx.z;
    int dim     = threadIdx.x;
    if (dim >= head_dim) return;

    int phys_id = block_ids[blk_idx];

    int64_t idx = (int64_t)phys_id * block_size * n_kv_heads * head_dim
                + (int64_t)slot   * n_kv_heads  * head_dim
                + (int64_t)head   * head_dim
                + dim;

    pool_k[idx] = __float2half(0.0f);
    pool_v[idx] = __float2half(0.0f);
}

__global__ void kv_write(
    __half* __restrict__ pool_k, __half* __restrict__ pool_v,
    const __half* __restrict__ k_in, const __half* __restrict__ v_in,
    const int* __restrict__ phys_blocks, const int* __restrict__ slots,
    int n_kv_heads, int head_dim, int block_size
) {
    int tok  = blockIdx.x;
    int head = blockIdx.y;
    int dim  = threadIdx.x;
    if (dim >= head_dim) return;

    int phys_id = phys_blocks[tok];
    int slot    = slots[tok];

    int64_t src = (int64_t)tok  * n_kv_heads * head_dim
                + (int64_t)head * head_dim
                + dim;

    int64_t dst = (int64_t)phys_id * block_size * n_kv_heads * head_dim
                + (int64_t)slot    * n_kv_heads  * head_dim
                + (int64_t)head    * head_dim
                + dim;

    pool_k[dst] = k_in[src];
    pool_v[dst] = v_in[src];
}

__global__ void kv_read(
    const __half* __restrict__ pool_k, const __half* __restrict__ pool_v,
    __half* __restrict__ k_out, __half* __restrict__ v_out,
    const int* __restrict__ phys_blocks, const int* __restrict__ slots,
    int n_kv_heads, int head_dim, int block_size
) {
    int tok  = blockIdx.x;
    int head = blockIdx.y;
    int dim  = threadIdx.x;
    if (dim >= head_dim) return;

    int phys_id = phys_blocks[tok];
    int slot    = slots[tok];

    int64_t src = (int64_t)phys_id * block_size * n_kv_heads * head_dim
                + (int64_t)slot    * n_kv_heads  * head_dim
                + (int64_t)head    * head_dim
                + dim;

    int64_t dst = (int64_t)tok  * n_kv_heads * head_dim
                + (int64_t)head * head_dim
                + dim;

    k_out[dst] = pool_k[src];
    v_out[dst] = pool_v[src];
}

// ─────────────────────────────────────────────────────────────────────────────
// extern "C" launchers
//
// Disables C++ name mangling so ctypes can resolve symbols by their plain names.
// All tensors arrive as void* (raw GPU device pointers from tensor.data_ptr()).
// ─────────────────────────────────────────────────────────────────────────────
extern "C" {

// Launch kv_init_blocks
// grid=(n_new, block_size, n_kv_heads)   block=(head_dim,)
void launch_init_blocks(
    void*       pool_k,
    void*       pool_v,
    const int*  block_ids,
    int n_new, int block_size, int n_kv_heads, int head_dim
) {
    dim3 grid(n_new, block_size, n_kv_heads);
    dim3 block(head_dim);
    kv_init_blocks<<<grid, block>>>(
        (__half*)pool_k, (__half*)pool_v,
        block_ids, block_size, n_kv_heads, head_dim
    );
}

// Launch kv_write
// grid=(T, n_kv_heads)   block=(head_dim,)
void launch_write(
    void*       pool_k,
    void*       pool_v,
    const void* k_in,
    const void* v_in,
    const int*  phys_blocks,
    const int*  slots,
    int T, int n_kv_heads, int head_dim, int block_size
) {
    dim3 grid(T, n_kv_heads);
    dim3 block(head_dim);
    kv_write<<<grid, block>>>(
        (__half*)pool_k,       (__half*)pool_v,
        (const __half*)k_in,   (const __half*)v_in,
        phys_blocks, slots,
        n_kv_heads, head_dim, block_size
    );
}

// Launch kv_read
// grid=(T, n_kv_heads)   block=(head_dim,)
void launch_read(
    const void* pool_k,
    const void* pool_v,
    void*       k_out,
    void*       v_out,
    const int*  phys_blocks,
    const int*  slots,
    int T, int n_kv_heads, int head_dim, int block_size
) {
    dim3 grid(T, n_kv_heads);
    dim3 block(head_dim);
    kv_read<<<grid, block>>>(
        (const __half*)pool_k, (const __half*)pool_v,
        (__half*)k_out,        (__half*)v_out,
        phys_blocks, slots,
        n_kv_heads, head_dim, block_size
    );
}

}

"""

CU_PATH = '/tmp/kv_kernels.cu'
SO_PATH = '/tmp/kv_ops.so'

with open(CU_PATH, 'w') as f:
    f.write(CUDA_SRC)
print(f'Written  : {CU_PATH}')

# ── Compile with nvcc ─────────────────────────────────────────────────────────
# Detect SM arch from the current GPU so we don't hardcode sm_75
major, minor = torch.cuda.get_device_capability(0)
arch = f'-arch=sm_{major}{minor}'
print(f'SM arch  : {major}.{minor}  ->  {arch}')

nvcc_cmd = [
    'nvcc', arch,
    '-O2', '--use_fast_math',
    '-shared',             # produce a .so shared library
    '-Xcompiler', '-fPIC', # position-independent code — required for .so
    '-o', SO_PATH,
    CU_PATH,
]
print(f'Running  : {" ".join(nvcc_cmd)}')
result = subprocess.run(nvcc_cmd, capture_output=True, text=True)
if result.returncode != 0:
    print('nvcc stderr:')
    print(result.stderr)
    raise RuntimeError('CUDA compilation failed')
print(f'Compiled : {SO_PATH}  ({os.path.getsize(SO_PATH)//1024} KB)')

# ── Load .so with ctypes ──────────────────────────────────────────────────────
# RTLD_GLOBAL: expose CUDA runtime symbols globally so the .so can call
# cudaMemcpy etc. from other loaded libraries without symbol lookup errors
lib = ctypes.CDLL(SO_PATH, mode=ctypes.RTLD_GLOBAL)

# ── Declare argtypes ─────────────────────────────────────────────────────────
# Without argtypes, ctypes passes Python ints as 64-bit values which
# silently corrupts the int parameters on the C side.
# void* (GPU pointer) -> c_void_p
# int                 -> c_int

lib.launch_init_blocks.argtypes = [
    ctypes.c_void_p, ctypes.c_void_p,   # pool_k, pool_v
    ctypes.c_void_p,                     # block_ids (int* on device)
    ctypes.c_int,                        # n_new
    ctypes.c_int,                        # block_size
    ctypes.c_int,                        # n_kv_heads
    ctypes.c_int,                        # head_dim
]
lib.launch_init_blocks.restype = None

lib.launch_write.argtypes = [
    ctypes.c_void_p, ctypes.c_void_p,   # pool_k, pool_v
    ctypes.c_void_p, ctypes.c_void_p,   # k_in, v_in
    ctypes.c_void_p, ctypes.c_void_p,   # phys_blocks, slots
    ctypes.c_int,                        # T
    ctypes.c_int,                        # n_kv_heads
    ctypes.c_int,                        # head_dim
    ctypes.c_int,                        # block_size
]
lib.launch_write.restype = None

lib.launch_read.argtypes = [
    ctypes.c_void_p, ctypes.c_void_p,   # pool_k, pool_v
    ctypes.c_void_p, ctypes.c_void_p,   # k_out, v_out
    ctypes.c_void_p, ctypes.c_void_p,   # phys_blocks, slots
    ctypes.c_int,                        # T
    ctypes.c_int,                        # n_kv_heads
    ctypes.c_int,                        # head_dim
    ctypes.c_int,                        # block_size
]
lib.launch_read.restype = None

print('Loaded   : kv_ops.so via ctypes')
print('   lib.launch_init_blocks  ->  kv_init_blocks kernel')
print('   lib.launch_write        ->  kv_write kernel')
print('   lib.launch_read         ->  kv_read kernel')


# ── Python wrappers ───────────────────────────────────────────────────────────
# tensor.data_ptr() returns the raw GPU device address as a Python int.
# Wrapping in c_void_p passes it to C as void* without any copy.

def _ptr(t: torch.Tensor) -> ctypes.c_void_p:
    """Raw GPU device pointer of a contiguous CUDA tensor."""
    return ctypes.c_void_p(t.data_ptr())


def init_blocks_cuda(pool_k: torch.Tensor, pool_v: torch.Tensor,
                     block_ids: List[int]):
    """
    Zero-fill physical blocks in pool_k and pool_v.
    Kernel kv_init_blocks: grid=(n, BLOCK_SIZE, N_KV_HEADS)  block=(HEAD_DIM,)
    """
    if not block_ids:
        return
    ids_t = torch.tensor(block_ids, dtype=torch.int32, device=device).contiguous()
    lib.launch_init_blocks(
        _ptr(pool_k), _ptr(pool_v),
        _ptr(ids_t),
        ctypes.c_int(len(block_ids)),
        ctypes.c_int(int(pool_k.shape[1])),   # BLOCK_SIZE
        ctypes.c_int(int(pool_k.shape[2])),   # N_KV_HEADS
        ctypes.c_int(int(pool_k.shape[3])),   # HEAD_DIM
    )


def write_kv_cuda(
    pool_k: torch.Tensor, pool_v: torch.Tensor,
    k_in: torch.Tensor,   v_in: torch.Tensor,   # (T, N_KV_HEADS, HEAD_DIM) fp16
    phys_blocks: torch.Tensor,                   # (T,) int32
    slots: torch.Tensor,                         # (T,) int32
):
    """
    Scatter K/V into paged pool slots.
    Kernel kv_write: grid=(T, N_KV_HEADS)  block=(HEAD_DIM,)
    """
    T = int(k_in.shape[0])
    if T == 0:
        return
    k_c  = k_in.contiguous().half()
    v_c  = v_in.contiguous().half()
    pb   = phys_blocks.contiguous().int()
    sl   = slots.contiguous().int()
    lib.launch_write(
        _ptr(pool_k), _ptr(pool_v),
        _ptr(k_c), _ptr(v_c),
        _ptr(pb), _ptr(sl),
        ctypes.c_int(T),
        ctypes.c_int(int(pool_k.shape[2])),   # N_KV_HEADS
        ctypes.c_int(int(pool_k.shape[3])),   # HEAD_DIM
        ctypes.c_int(int(pool_k.shape[1])),   # BLOCK_SIZE
    )


def read_kv_cuda(
    pool_k: torch.Tensor, pool_v: torch.Tensor,
    phys_blocks: torch.Tensor,                   # (T,) int32
    slots: torch.Tensor,                         # (T,) int32
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Gather K/V from paged pool into fresh contiguous buffers.
    Kernel kv_read: grid=(T, N_KV_HEADS)  block=(HEAD_DIM,)
    Returns (k_out, v_out) each (T, N_KV_HEADS, HEAD_DIM) fp16.
    """
    T   = int(phys_blocks.shape[0])
    NKV = int(pool_k.shape[2])
    HD  = int(pool_k.shape[3])
    k_out = torch.empty((T, NKV, HD), dtype=torch.float16, device=device)
    v_out = torch.empty_like(k_out)
    if T == 0:
        return k_out, v_out
    pb  = phys_blocks.contiguous().int()
    sl  = slots.contiguous().int()
    lib.launch_read(
        _ptr(pool_k), _ptr(pool_v),
        _ptr(k_out), _ptr(v_out),
        _ptr(pb), _ptr(sl),
        ctypes.c_int(T),
        ctypes.c_int(NKV),
        ctypes.c_int(HD),
        ctypes.c_int(int(pool_k.shape[1])),   # BLOCK_SIZE
    )
    return k_out, v_out


def clear_blocks_cuda(pool_k: torch.Tensor, pool_v: torch.Tensor,
                      block_ids: List[int]):
    """
    Zero-fill freed blocks — reuses kv_init_blocks kernel.
    Prevents stale K/V from leaking into future sequences.
    """
    init_blocks_cuda(pool_k, pool_v, block_ids)


Written  : /tmp/kv_kernels.cu
SM arch  : 7.5  ->  -arch=sm_75
Running  : nvcc -arch=sm_75 -O2 --use_fast_math -shared -Xcompiler -fPIC -o /tmp/kv_ops.so /tmp/kv_kernels.cu
Compiled : /tmp/kv_ops.so  (983 KB)
Loaded   : kv_ops.so via ctypes
   lib.launch_init_blocks  ->  kv_init_blocks kernel
   lib.launch_write        ->  kv_write kernel
   lib.launch_read         ->  kv_read kernel


In [8]:
# ── Round-trip test: write_kv → read_kv → clear ──────────────────────────────
print('=== Round-trip test: write_kv → read_kv ===')

T_test   = 35
k_test   = torch.randn(T_test, cfg.n_kv_heads, cfg.head_dim, device=device, dtype=torch.float16)
v_test   = torch.randn_like(k_test)

# Manual block/slot assignment for 35 tokens
# Tokens 0-15  → block 0, slots 0-15
# Tokens 16-31 → block 1, slots 0-15
# Tokens 32-34 → block 2, slots 0-2
phys_ids = torch.tensor([0]*16 + [1]*16 + [2]*3, dtype=torch.int32, device=device)
slot_ids = torch.tensor(list(range(16)) + list(range(16)) + list(range(3)),
                        dtype=torch.int32, device=device)

# ── 1. Init blocks (zero-fill) ───────────────────────────────────────────────
pool_k[0:3] = 99.0   # poison first
init_blocks_cuda(pool_k, pool_v, [0, 1, 2])
torch.cuda.synchronize()
assert pool_k[0].abs().max().item() == 0.0, 'init failed'
print('kv_init_blocks  blocks zeroed')

# ── 2. Write K/V ─────────────────────────────────────────────────────────────
write_kv_cuda(pool_k, pool_v, k_test, v_test, phys_ids, slot_ids)
torch.cuda.synchronize()

# ── 3. Read back ─────────────────────────────────────────────────────────────
k_read, v_read = read_kv_cuda(pool_k, pool_v, phys_ids, slot_ids)
torch.cuda.synchronize()

k_ok = torch.allclose(k_test, k_read, atol=1e-3)
v_ok = torch.allclose(v_test, v_read, atol=1e-3)
print(f'kv_write        K match={k_ok}  max_err={(k_test-k_read).abs().max().item():.5f}')
print(f'kv_read         V match={v_ok}  max_err={(v_test-v_read).abs().max().item():.5f}')
assert k_ok and v_ok, 'Round-trip failed!'

# ── 4. Clear (reuses init kernel) ────────────────────────────────────────────
clear_blocks_cuda(pool_k, pool_v, [0, 1, 2])
torch.cuda.synchronize()
k_cleared, _ = read_kv_cuda(pool_k, pool_v, phys_ids, slot_ids)
assert k_cleared.abs().max().item() == 0.0, 'clear failed!'
print('kv_clear_blocks blocks zeroed after free')
print()
print('All 4 .cu CUDA kernels working correctly')


=== Round-trip test: write_kv → read_kv ===
kv_init_blocks  blocks zeroed
kv_write        K match=True  max_err=0.00000
kv_read         V match=True  max_err=0.00000
kv_clear_blocks blocks zeroed after free

All 4 .cu CUDA kernels working correctly


---
## 6. GPUPagedKVCache — Contiguous Pool with CUDA Kernels

Drop-in replacement for notebook 02's dict-based `PagedKVCache`.
Same API, but backed by the contiguous GPU pool and CUDA kernels.

In [9]:
from collections import deque

class BlockAllocator:
    """Same as notebook 02 — free-list over integer block ids."""
    def __init__(self, n_total):
        self.n_total    = n_total
        self._free      = deque(range(n_total))
        self._ref_count = [0] * n_total

    def allocate(self, n=1):
        if len(self._free) < n:
            raise RuntimeError(f'Out of KV blocks! {len(self._free)} free, {n} requested')
        ids = [self._free.popleft() for _ in range(n)]
        for b in ids: self._ref_count[b] = 1
        return ids

    def free(self, ids):
        for b in ids:
            self._ref_count[b] -= 1
            if self._ref_count[b] == 0:
                self._free.append(b)

    @property
    def n_free(self): return len(self._free)
    @property
    def n_used(self): return self.n_total - self.n_free
    def __repr__(self):
        return f'BlockAllocator(total={self.n_total}, free={self.n_free}, used={self.n_used})'


class LayeredBlockTable:
    """Same as notebook 02 — per-layer physical block ids."""
    def __init__(self, seq_id, n_layers, block_size=BLOCK_SIZE):
        self.seq_id = seq_id; self.n_layers = n_layers
        self.block_size = block_size; self.num_tokens = 0
        self.block_ids: List[List[int]] = [[] for _ in range(n_layers)]

    @property
    def num_blocks(self): return len(self.block_ids[0])
    @property
    def last_block_is_full(self):
        return self.num_tokens > 0 and (self.num_tokens % self.block_size) == 0
    @property
    def last_block_offset(self): return self.num_tokens % self.block_size

    def append_token(self, allocator, kv_cache=None):
        if self.num_blocks == 0 or self.last_block_is_full:
            new_ids = allocator.allocate(self.n_layers)
            for li in range(self.n_layers):
                self.block_ids[li].append(new_ids[li])
                if kv_cache is not None:
                    kv_cache.on_block_allocated(li, new_ids[li])
        self.num_tokens += 1

    def append_tokens(self, n, allocator, kv_cache=None):
        for _ in range(n): self.append_token(allocator, kv_cache)

    def translate(self, layer, logical_pos):
        lb = logical_pos // self.block_size
        so = logical_pos %  self.block_size
        return self.block_ids[layer][lb], so

    def get_all_physical_slots(self, layer):
        pbs, ss = [], []
        for pos in range(self.num_tokens):
            pb, s = self.translate(layer, pos)
            pbs.append(pb); ss.append(s)
        return pbs, ss

    def free(self, allocator, kv_cache=None):
        all_ids = [b for layer_ids in self.block_ids for b in layer_ids]
        if kv_cache is not None:
            kv_cache.on_blocks_freed(all_ids)
        allocator.free(all_ids)
        self.block_ids = [[] for _ in range(self.n_layers)]
        self.num_tokens = 0

    def __repr__(self):
        return (f'LayeredBlockTable(seq={self.seq_id}, tokens={self.num_tokens}, '
                f'blocks_per_layer={self.num_blocks}, layer0={self.block_ids[0]})')


class GPUPagedKVCache:
    """
    Paged KV cache backed by a single contiguous GPU tensor.

    pool_k : (N_PHYSICAL_BLOCKS, BLOCK_SIZE, N_KV_HEADS, HEAD_DIM)  fp16
    pool_v : same

    All reads/writes go through Triton CUDA kernels.
    Block allocation/deallocation triggers kernel_init / kernel_clear.
    """

    def __init__(self, pool_k: torch.Tensor, pool_v: torch.Tensor, n_layers: int):
        assert pool_k.is_contiguous() and pool_v.is_contiguous()
        assert pool_k.dtype == torch.float16
        self.pool_k   = pool_k
        self.pool_v   = pool_v
        self.n_layers = n_layers
        self.n_phys   = pool_k.shape[0]
        self.block_size = pool_k.shape[1]
        self.n_kv_heads = pool_k.shape[2]
        self.head_dim   = pool_k.shape[3]

        pool_mb = pool_k.numel() * 2 * 2 / 1e6   # K+V, fp16
        print(f'GPUPagedKVCache ready')
        print(f'  Pool shape  : {list(pool_k.shape)}  × K+V')
        print(f'  Pool size   : {pool_mb:.1f} MB  (contiguous GPU memory)')
        print(f'  N physical  : {self.n_phys} blocks')
        print(f'  N logical   : {self.n_phys // n_layers} blocks')

    # ── Lifecycle callbacks (called by LayeredBlockTable) ────────────────────
    def on_block_allocated(self, layer: int, block_id: int):
        """Called when LayeredBlockTable gets a new physical block for a layer."""
        # Zero-fill this single block using the init kernel
        init_blocks_cuda(self.pool_k, self.pool_v, [block_id])

    def on_blocks_freed(self, block_ids: List[int]):
        """Called when a sequence is freed — clear all its blocks."""
        clear_blocks_cuda(self.pool_k, self.pool_v, block_ids)

    # ── Write: called by PagedLLaMAAttention during forward ──────────────────
    def write_tokens(
        self,
        layer      : int,
        block_table: LayeredBlockTable,
        k_seq      : torch.Tensor,   # (T, n_kv_heads, head_dim)
        v_seq      : torch.Tensor,
        start_pos  : int = 0,
    ):
        T = k_seq.shape[0]
        phys_list, slot_list = [], []
        for i in range(T):
            pb, s = block_table.translate(layer, start_pos + i)
            phys_list.append(pb); slot_list.append(s)

        phys_t = torch.tensor(phys_list, dtype=torch.int32, device=device)
        slot_t = torch.tensor(slot_list, dtype=torch.int32, device=device)
        write_kv_cuda(self.pool_k, self.pool_v, k_seq, v_seq, phys_t, slot_t)

    # ── Read: called by paged_attention kernel ────────────────────────────────
    def read_sequence(
        self,
        layer      : int,
        block_table: LayeredBlockTable,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        phys_list, slot_list = block_table.get_all_physical_slots(layer)
        phys_t = torch.tensor(phys_list, dtype=torch.int32, device=device)
        slot_t = torch.tensor(slot_list, dtype=torch.int32, device=device)
        return read_kv_cuda(self.pool_k, self.pool_v, phys_t, slot_t)

    def memory_used_mb(self):
        return self.pool_k.numel() * 2 * 2 / 1e6   # always full pool

    def __repr__(self):
        return (f'GPUPagedKVCache(phys_blocks={self.n_phys}, '
                f'logical_blocks={self.n_phys//self.n_layers}, '
                f'pool={self.memory_used_mb():.1f} MB)')


# ── Instantiate ──────────────────────────────────────────────────────────────
gpu_kv_cache = GPUPagedKVCache(pool_k, pool_v, cfg.n_layers)
allocator    = BlockAllocator(N_PHYSICAL_BLOCKS)
print(f'\nAllocator: {allocator}')

GPUPagedKVCache ready
  Pool shape  : [560436, 16, 6, 64]  × K+V
  Pool size   : 13773.3 MB  (contiguous GPU memory)
  N physical  : 560436 blocks
  N logical   : 46703 blocks

Allocator: BlockAllocator(total=560436, free=560436, used=0)


---
## 7. Integration Test

Verify that `GPUPagedKVCache` works end-to-end with `LayeredBlockTable` — write K/V for a sequence token by token, then read it back.

In [10]:
print('=== Integration Test: LayeredBlockTable + GPUPagedKVCache ===')

SEQ_LEN = 48   # 3 full blocks of 16

# Allocate a sequence
bt_int = LayeredBlockTable(seq_id=0, n_layers=cfg.n_layers)
bt_int.append_tokens(SEQ_LEN, allocator, gpu_kv_cache)

print(f'Sequence of {SEQ_LEN} tokens:')
print(f'  Logical blocks      : {bt_int.num_blocks}')
print(f'  Allocator slots used: {allocator.n_used}  '
      f'(= {bt_int.num_blocks} × {cfg.n_layers} layers)')
print(f'  Layer 0 block ids   : {bt_int.block_ids[0]}')
print(f'  Layer 1 block ids   : {bt_int.block_ids[1]}  ← different from layer 0')

# Write fake K/V for layer 0
k_fake = torch.randn(SEQ_LEN, cfg.n_kv_heads, cfg.head_dim, device=device, dtype=torch.float16)
v_fake = torch.randn(SEQ_LEN, cfg.n_kv_heads, cfg.head_dim, device=device, dtype=torch.float16)
gpu_kv_cache.write_tokens(layer=0, block_table=bt_int, k_seq=k_fake, v_seq=v_fake, start_pos=0)

# Read back layer 0
k_out, v_out = gpu_kv_cache.read_sequence(layer=0, block_table=bt_int)
torch.cuda.synchronize()

k_ok = torch.allclose(k_fake, k_out, atol=1e-3)
v_ok = torch.allclose(v_fake, v_out, atol=1e-3)
print(f'\nLayer 0 K roundtrip: {k_ok}  max_err={(k_fake-k_out).abs().max().item():.5f}')
print(f'Layer 0 V roundtrip: {v_ok}  max_err={(v_fake-v_out).abs().max().item():.5f}')

# Layer 1 should be all zeros (never written)
k_l1, _ = gpu_kv_cache.read_sequence(layer=1, block_table=bt_int)
assert k_l1.abs().max().item() == 0.0, 'Layer isolation broken!'
print(f'Layer 1 isolation  :  (zeros — never written)')

# Free sequence — should zero out its blocks
bt_int.free(allocator, gpu_kv_cache)
torch.cuda.synchronize()
print(f'\nAfter free: allocator={allocator}')
assert allocator.n_used == 0, 'Blocks not freed!'
print('Integration test passed')

=== Integration Test: LayeredBlockTable + GPUPagedKVCache ===
Sequence of 48 tokens:
  Logical blocks      : 3
  Allocator slots used: 36  (= 3 × 12 layers)
  Layer 0 block ids   : [0, 12, 24]
  Layer 1 block ids   : [1, 13, 25]  ← different from layer 0

Layer 0 K roundtrip: True  max_err=0.00000
Layer 0 V roundtrip: True  max_err=0.00000
Layer 1 isolation  :  (zeros — never written)

After free: allocator=BlockAllocator(total=560436, free=560436, used=0)
Integration test passed


---
## 8. Benchmarks — Kernel Throughput & Memory Bandwidth

In [11]:
import time

torch.cuda.synchronize()

N_POOL = int(pool_k.shape[0])   # actual allocated size — safe upper bound

def benchmark_kernel(fn, n_warmup=5, n_repeat=50, label=''):
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_repeat):
        fn()
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    ms = (t1 - t0) / n_repeat * 1000
    print(f'  {label:<40}: {ms:.3f} ms/call')
    return ms

print('=== Kernel Benchmarks ===')
print(f'Config: n_kv_heads={cfg.n_kv_heads}, head_dim={cfg.head_dim}, '
      f'block_size={BLOCK_SIZE}, N_POOL={N_POOL}')
print()

# ── Bench write_kv ───────────────────────────────────────────────────────────
for T_bench in [16, 64, 256, 512]:
    k_b = torch.randn(T_bench, cfg.n_kv_heads, cfg.head_dim,
                      dtype=torch.float16, device=device)
    v_b = torch.randn_like(k_b)
    pb  = torch.randint(0, N_POOL,     (T_bench,), dtype=torch.int32, device=device)
    sl  = torch.randint(0, BLOCK_SIZE, (T_bench,), dtype=torch.int32, device=device)
    fn  = lambda k=k_b, v=v_b, p=pb, s=sl: write_kv_cuda(pool_k, pool_v, k, v, p, s)
    ms  = benchmark_kernel(fn, label=f'write_kv  T={T_bench}')
    bw  = (T_bench * cfg.n_kv_heads * cfg.head_dim * 2 * 2) / (ms * 1e-3) / 1e9
    print(f'    → {bw:.1f} GB/s effective write bandwidth')

print()

# ── Bench read_kv ─────────────────────────────────────────────────────────────
for T_bench in [16, 64, 256, 512]:
    pb  = torch.randint(0, N_POOL,     (T_bench,), dtype=torch.int32, device=device)
    sl  = torch.randint(0, BLOCK_SIZE, (T_bench,), dtype=torch.int32, device=device)
    fn  = lambda p=pb, s=sl: read_kv_cuda(pool_k, pool_v, p, s)
    ms  = benchmark_kernel(fn, label=f'read_kv   T={T_bench}')
    bw  = (T_bench * cfg.n_kv_heads * cfg.head_dim * 2 * 2) / (ms * 1e-3) / 1e9
    print(f'    → {bw:.1f} GB/s effective read bandwidth')

print()

# ── Bench init_blocks ─────────────────────────────────────────────────────────
for n_blocks in [1, 8, 32]:
    ids = list(range(n_blocks))
    fn  = lambda i=ids: init_blocks_cuda(pool_k, pool_v, i)
    benchmark_kernel(fn, label=f'init_blocks n={n_blocks}')

print(f'\nT4 theoretical memory bandwidth: ~320 GB/s')
print(f'(scattered access is less efficient than sequential — expect 10-50% of peak)')

=== Kernel Benchmarks ===
Config: n_kv_heads=6, head_dim=64, block_size=16, N_POOL=560436

  write_kv  T=16                          : 0.011 ms/call
    → 2.3 GB/s effective write bandwidth
  write_kv  T=64                          : 0.010 ms/call
    → 9.9 GB/s effective write bandwidth
  write_kv  T=256                         : 0.010 ms/call
    → 37.7 GB/s effective write bandwidth
  write_kv  T=512                         : 0.012 ms/call
    → 64.0 GB/s effective write bandwidth

  read_kv   T=16                          : 0.021 ms/call
    → 1.2 GB/s effective read bandwidth
  read_kv   T=64                          : 0.020 ms/call
    → 4.8 GB/s effective read bandwidth
  read_kv   T=256                         : 0.021 ms/call
    → 19.0 GB/s effective read bandwidth
  read_kv   T=512                         : 0.019 ms/call
    → 41.4 GB/s effective read bandwidth

  init_blocks n=1                         : 0.033 ms/call
  init_blocks n=8                         : 0.033 ms/call

---
## 9. Stress Test — Fill Pool to Capacity

In [12]:
print('=== Stress Test: Fill pool to capacity ===')
print(f'Target: {N_LOGICAL_BLOCKS} logical blocks × {cfg.n_layers} layers = {N_PHYSICAL_BLOCKS} physical slots')

stress_alloc = BlockAllocator(N_PHYSICAL_BLOCKS)
stress_tables = []
seq_id = 0
n_errors = 0

# Fill pool with sequences of 32 tokens (2 logical blocks each)
TOKENS_PER_SEQ = 32
max_seqs = N_LOGICAL_BLOCKS // (TOKENS_PER_SEQ // BLOCK_SIZE)
print(f'Filling with {max_seqs} sequences of {TOKENS_PER_SEQ} tokens each...')

for i in range(max_seqs):
    try:
        bt = LayeredBlockTable(seq_id=i, n_layers=cfg.n_layers)
        bt.append_tokens(TOKENS_PER_SEQ, stress_alloc, gpu_kv_cache)
        stress_tables.append(bt)
    except RuntimeError as e:
        print(f'  OOM at seq {i}: {e}')
        n_errors += 1
        break

torch.cuda.synchronize()
free_during, _ = torch.cuda.mem_get_info()

print(f'\nAfter filling:')
print(f'  Sequences allocated  : {len(stress_tables)}')
print(f'  Logical blocks used  : {stress_alloc.n_used // cfg.n_layers}')
print(f'  Physical slots used  : {stress_alloc.n_used}')
print(f'  Physical slots free  : {stress_alloc.n_free}')
print(f'  GPU free memory      : {free_during/1e6:.0f} MB')
print(f'  OOM errors           : {n_errors}  (should be 0 — pool is pre-allocated)')

# Write K/V into all sequences (layer 0 only for speed)
print(f'\nWriting K/V for all sequences (layer 0)...')
t0 = time.perf_counter()
for bt in stress_tables:
    k_s = torch.randn(TOKENS_PER_SEQ, cfg.n_kv_heads, cfg.head_dim,
                      dtype=torch.float16, device=device)
    v_s = torch.randn_like(k_s)
    gpu_kv_cache.write_tokens(0, bt, k_s, v_s, start_pos=0)
torch.cuda.synchronize()
t1 = time.perf_counter()
print(f'  Write time: {(t1-t0)*1000:.1f} ms for {len(stress_tables)} sequences')

# Free all
print(f'\nFreeing all sequences...')
t0 = time.perf_counter()
for bt in stress_tables:
    bt.free(stress_alloc, gpu_kv_cache)
torch.cuda.synchronize()
t1 = time.perf_counter()

assert stress_alloc.n_used == 0, 'Memory leak after stress test!'
print(f'  Free time  : {(t1-t0)*1000:.1f} ms')
print(f'  Allocator  : {stress_alloc}')
print('\nStress test passed — no OOM, no memory leak')

# Final GPU state
torch.cuda.synchronize()
free_final, _ = torch.cuda.mem_get_info()
print(f'\n=== Final GPU Memory ===')
print(f'  Free: {free_final/1e6:.0f} MB  (pool is still reserved — dealloc with del pool_k/pool_v)')

=== Stress Test: Fill pool to capacity ===
Target: 46703 logical blocks × 12 layers = 560436 physical slots
Filling with 23351 sequences of 32 tokens each...

After filling:
  Sequences allocated  : 23351
  Logical blocks used  : 46702
  Physical slots used  : 560424
  Physical slots free  : 12
  GPU free memory      : 1499 MB
  OOM errors           : 0  (should be 0 — pool is pre-allocated)

Writing K/V for all sequences (layer 0)...
  Write time: 3002.7 ms for 23351 sequences

Freeing all sequences...
  Free time  : 1431.6 ms
  Allocator  : BlockAllocator(total=560436, free=560436, used=0)

Stress test passed — no OOM, no memory leak

=== Final GPU Memory ===
  Free: 1499 MB  (pool is still reserved — dealloc with del pool_k/pool_v)


In [13]:
print('=== Memory Summary ===')
print(f'  GPU                 : {props.name}')
print(f'  Total VRAM          : {total/1e9:.2f} GB')
print(f'  Model weights (fp16): {model_vram/1e6:.0f} MB')
print(f'  KV pool (K+V)       : {KV_POOL_BYTES/1e6:.0f} MB  ({N_LOGICAL_BLOCKS} logical blocks)')
print(f'  Activation reserve  : {ACTIVATION_RESERVE_GB*1e3:.0f} MB')
print(f'  Safety margin       : {SAFETY_MARGIN_GB*1e3:.0f} MB')
print(f'  Max tokens cached   : {N_LOGICAL_BLOCKS * BLOCK_SIZE:,}')
print(f'  Max seqs (32 tok ea): {N_LOGICAL_BLOCKS * BLOCK_SIZE // 32}')

=== Memory Summary ===
  GPU                 : Tesla T4
  Total VRAM          : 15.64 GB
  Model weights (fp16): 254 MB
  KV pool (K+V)       : 13773 MB  (46703 logical blocks)
  Activation reserve  : 1000 MB
  Safety margin       : 500 MB
  Max tokens cached   : 747,248
  Max seqs (32 tok ea): 23351


---
## 10. End-to-End Inference with Paged KV Cache

Puts everything together: the loaded LLaMA weights + CUDA paged pool + autoregressive decode.

```
prompt
  │
  ▼ tokenizer.encode()
  │
  ▼ PREFILL — full prompt in one forward pass
  │   for each layer:
  │     wk(x) → K,  wv(x) → V
  │     kv_write kernel → pool_k[phys_id, slot, :, :] = K
  │
  ▼ DECODE — one token at a time
  │   for each step:
  │     allocate slot for new token
  │     wk(x) → K,  wv(x) → V  →  kv_write
  │     kv_read gathers ALL past K,V
  │     attention(Q, K_all, V_all) → next token
  │
  ▼ bt.free() → clear_blocks kernel → blocks back to allocator
```

In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# 10. End-to-End Inference with Paged KV Cache
#
# Pipeline:
#   prompt  →  tokenizer  →  PagedLLaMA (forward with GPUPagedKVCache)
#           →  autoregressive decode (one token at a time)
#           →  each new token's K/V written to paged pool via CUDA kernels
#           →  attention gathers K/V from paged pool
#           →  generated text
# ═══════════════════════════════════════════════════════════════════════════

# ── 1. Build a full forward-capable LLaMA on top of the loaded weights ───────
# LLaMAModelWeightsOnly (cell 5) has the weights but no forward().
# We build PagedLLaMAInfer which:
#   - shares the same weight tensors (no extra VRAM)
#   - runs a proper forward pass
#   - writes K/V to GPUPagedKVCache via the CUDA kernels
#   - reads K/V back for attention via the CUDA kernels
TOKENIZER_ID = 'huggyllama/llama-7b'
print(f'Loading tokenizer: {TOKENIZER_ID}')
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
tokenizer.pad_token = tokenizer.eos_token

class PagedLLaMAInfer(nn.Module):
    """
    Full inference-capable LLaMA that uses GPUPagedKVCache.

    Weights are copied from the already-loaded LLaMAModelWeightsOnly
    so no extra VRAM is used for parameters.

    Forward pass per layer:
      1. Project x → Q, K, V
      2. Apply RoPE to Q and K
      3. Write K, V to GPUPagedKVCache (CUDA kv_write kernel)
      4. Read ALL past K, V from GPUPagedKVCache (CUDA kv_read kernel)
      5. Attention(Q, K_all, V_all)
      6. Output projection + residual
      7. SwiGLU FFN + residual
    """

    def __init__(self, weights_model: 'LLaMAModelWeightsOnly', cfg: LLaMAConfig):
        super().__init__()
        self.cfg = cfg

        # ── Share weight tensors — no copy, no extra VRAM ────────────────────
        self.token_emb = weights_model.token_emb
        self.layers    = weights_model.layers
        self.norm      = weights_model.norm
        self.lm_head   = weights_model.lm_head

        # Causal mask (additive, upper-tri = -inf)
        mask = torch.full((1, 1, cfg.max_seq_len, cfg.max_seq_len), float('-inf'))
        mask = torch.triu(mask, diagonal=1).to(device)
        self.register_buffer('causal_mask', mask)

    def _attn_layer_forward(
        self,
        x           : torch.Tensor,      # (1, T, dim)
        layer_idx   : int,
        block_table : LayeredBlockTable,
        kv_cache    : GPUPagedKVCache,
        start_pos   : int,
    ) -> torch.Tensor:
        """One attention sublayer with paged KV read/write."""
        B, T, _ = x.shape
        attn = self.layers[layer_idx].attention
        cfg  = self.cfg

        # ── Project Q, K, V ──────────────────────────────────────────────────
        q = attn.wq(x).view(B, T, cfg.n_heads,    cfg.head_dim).transpose(1, 2)
        k = attn.wk(x).view(B, T, cfg.n_kv_heads, cfg.head_dim).transpose(1, 2)
        v = attn.wv(x).view(B, T, cfg.n_kv_heads, cfg.head_dim).transpose(1, 2)

        # ── Apply RoPE ───────────────────────────────────────────────────────
        cos = cos_table[start_pos : start_pos + T].to(x.device)
        sin = sin_table[start_pos : start_pos + T].to(x.device)
        q, k = apply_rope(q, k, cos, sin)

        # ── Write K, V to paged pool (CUDA kv_write kernel) ──────────────────
        # k shape: (1, n_kv_heads, T, head_dim) → (T, n_kv_heads, head_dim)
        k_write = k.squeeze(0).permute(1, 0, 2).contiguous()
        v_write = v.squeeze(0).permute(1, 0, 2).contiguous()
        kv_cache.write_tokens(
            layer=layer_idx, block_table=block_table,
            k_seq=k_write, v_seq=v_write, start_pos=start_pos
        )

        # ── Read ALL past K, V from paged pool (CUDA kv_read kernel) ─────────
        # Returns (T_total, n_kv_heads, head_dim) — all tokens seen so far
        k_all, v_all = kv_cache.read_sequence(layer=layer_idx, block_table=block_table)
        T_total = k_all.shape[0]

        # Reshape to (1, n_kv_heads, T_total, head_dim) for attention
        k_all = k_all.permute(1, 0, 2).unsqueeze(0)   # (1, nkv, T_total, hd)
        v_all = v_all.permute(1, 0, 2).unsqueeze(0)

        # ── GQA: expand K/V heads to match Q heads ────────────────────────────
        k_all = repeat_kv(k_all, cfg.n_heads // cfg.n_kv_heads)
        v_all = repeat_kv(v_all, cfg.n_heads // cfg.n_kv_heads)

        # ── Scaled dot-product attention ──────────────────────────────────────
        scale  = 1.0 / math.sqrt(cfg.head_dim)
        scores = torch.matmul(q, k_all.transpose(-2, -1)) * scale  # (1,nh,T,T_total)

        # Apply causal mask (only during prefill T>1; decode T=1 sees all past)
        if T > 1:
            scores = scores + self.causal_mask[:, :, start_pos:start_pos+T, :T_total].to(scores.device)

        weights = F.softmax(scores.float(), dim=-1).type_as(q)
        out     = torch.matmul(weights, v_all)                      # (1,nh,T,hd)
        out     = out.transpose(1, 2).contiguous().view(B, T, -1)   # (1,T,dim)
        return attn.wo(out)

    @torch.no_grad()
    def forward(
        self,
        idx         : torch.Tensor,        # (1, T) token ids
        block_table : LayeredBlockTable,
        kv_cache    : GPUPagedKVCache,
        start_pos   : int = 0,
    ) -> torch.Tensor:                     # returns logits (1, T, vocab_size)
        B, T = idx.shape
        assert B == 1

        x = self.token_emb(idx)            # (1, T, dim)

        for li, layer in enumerate(self.layers):
            # ── Attention sublayer (paged) ────────────────────────────────────
            h = self._attn_layer_forward(
                layer.attention_norm(x),
                layer_idx=li,
                block_table=block_table,
                kv_cache=kv_cache,
                start_pos=start_pos,
            )
            x = x + h

            # ── FFN sublayer ──────────────────────────────────────────────────
            x = x + layer.ffn(layer.ffn_norm(x))

        x = self.norm(x)
        return self.lm_head(x)             # (1, T, vocab_size)


# ── Instantiate — shares weights with already-loaded model, zero extra VRAM ──
paged_llama = PagedLLaMAInfer(model, cfg)
paged_llama.eval()
print(f'PagedLLaMAInfer ready — shares weights with loaded model')
print(f'Parameters: {sum(p.numel() for p in paged_llama.parameters())/1e6:.1f}M  '
      f'(same tensors, no copy)')


# ── 2. Generation function ────────────────────────────────────────────────────
@torch.no_grad()
def generate_paged(
    prompt         : str,
    max_new_tokens : int   = 100,
    temperature    : float = 0.8,
    top_k          : int   = 50,
) -> str:
    """
    Autoregressive generation using GPUPagedKVCache.

    Prefill  : full prompt processed in one forward pass
               → K/V for all prompt tokens written to paged pool
    Decode   : one token at a time
               → each step writes ONE new K/V to the pool
               → attention reads ALL past K/V from pool via kv_read kernel
    """
    # ── Fresh allocator + KV cache for this sequence ──────────────────────────
    gen_allocator = BlockAllocator(N_PHYSICAL_BLOCKS)
    gen_kv        = GPUPagedKVCache(pool_k, pool_v, cfg.n_layers)
    bt            = LayeredBlockTable(seq_id=0, n_layers=cfg.n_layers)

    tokens  = tokenizer.encode(prompt, add_special_tokens=True)
    T_start = len(tokens)

    print(f'Prompt tokens  : {T_start}')
    free_before, _ = torch.cuda.mem_get_info()

    # ── Prefill ───────────────────────────────────────────────────────────────
    # Allocate blocks for all prompt tokens upfront
    bt.append_tokens(T_start, gen_allocator, gen_kv)

    idx = torch.tensor([tokens], dtype=torch.long, device=device)
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        logits = paged_llama(idx, bt, gen_kv, start_pos=0)

    # Sample first generated token
    next_tok = _sample(logits[0, -1, :], temperature, top_k)
    tokens.append(next_tok)

    print(f'Prefill done   : {bt.num_tokens} slots used, '
          f'{bt.num_blocks} logical blocks × {cfg.n_layers} layers')
    free_after_prefill, _ = torch.cuda.mem_get_info()
    kv_used = (free_before - free_after_prefill) / 1e6
    print(f'KV pool used   : ~{kv_used:.1f} MB after prefill')
    print(f'Generating...')

    # ── Decode ────────────────────────────────────────────────────────────────
    for i in range(max_new_tokens - 1):
        # Allocate a slot for the NEW token BEFORE the forward pass
        bt.append_token(gen_allocator, gen_kv)
        start_pos = bt.num_tokens - 1

        idx_new = torch.tensor([[next_tok]], dtype=torch.long, device=device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = paged_llama(idx_new, bt, gen_kv, start_pos=start_pos)

        next_tok = _sample(logits[0, -1, :], temperature, top_k)
        tokens.append(next_tok)

        if next_tok == tokenizer.eos_token_id:
            print(f'EOS at step {i+1}')
            break

    n_generated = len(tokens) - T_start
    print(f'Generated      : {n_generated} tokens')
    print(f'Logical blocks : {bt.num_blocks}  '
          f'(allocator slots used: {gen_allocator.n_used})')

    # ── Free all blocks ───────────────────────────────────────────────────────
    bt.free(gen_allocator, gen_kv)
    torch.cuda.synchronize()
    print(f'Blocks freed   : allocator back to {gen_allocator}')

    return tokenizer.decode(tokens, skip_special_tokens=True)


def _sample(
    logits     : torch.Tensor,
    temperature: float = 0.8,
    top_k      : int   = 50,
) -> int:
    logits = logits.float() / max(temperature, 1e-5)
    if top_k > 0:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[-1]] = float('-inf')
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1).item()


# ── 3. Run inference ──────────────────────────────────────────────────────────
prompts = [
    'Once upon a time there was a little dragon who',
    'The princess looked at the knight and said',
]

for prompt in prompts:
    print('\n' + '='*65)
    print(f'PROMPT: {prompt!r}')
    print('='*65)
    t0 = time.time()
    output = generate_paged(prompt, max_new_tokens=80, temperature=0.8, top_k=50)
    elapsed = time.time() - t0
    n_new = len(tokenizer.encode(output)) - len(tokenizer.encode(prompt))
    print(f'Speed          : {n_new/elapsed:.1f} tok/s')
    print(f'\nOUTPUT:\n{output}')

print('\n' + '='*65)
print('All inference complete — KV cache backed by CUDA paged pool')
free_final, _ = torch.cuda.mem_get_info()
print(f'GPU free after all generations: {free_final/1e9:.2f} GB '
      f'(blocks freed, pool still reserved)')


Loading tokenizer: huggyllama/llama-7b
PagedLLaMAInfer ready — shares weights with loaded model
Parameters: 109.5M  (same tensors, no copy)

PROMPT: 'Once upon a time there was a little dragon who'
GPUPagedKVCache ready
  Pool shape  : [560436, 16, 6, 64]  × K+V
  Pool size   : 13773.3 MB  (contiguous GPU memory)
  N physical  : 560436 blocks
  N logical   : 46703 blocks
Prompt tokens  : 12
Prefill done   : 12 slots used, 1 logical blocks × 12 layers
KV pool used   : ~2.1 MB after prefill
Generating...
Generated      : 80 tokens
Logical blocks : 6  (allocator slots used: 72)
Blocks freed   : allocator back to BlockAllocator(total=560436, free=560436, used=0)
Speed          : 14.2 tok/s

OUTPUT:
Once upon a time there was a little dragon who liked playing a boat. He had a bucket in his eye. He was a strongummy blanket and always laughed like his kay.

The little boy was so excited to play. He ran around the house and yelled at the kite. But when he got there, he noticed something strang